# 09 - Calibration: from classifier output to trust score

**CPU is fine. Run all; idempotent** (recomputes everything from saved predictions).

A raw classifier probability is not a trust score. Calibration on the
validation part - disjoint from both training and test - is what licenses
reading the number as a probability, and therefore what licenses setting
operating thresholds by cost.

What this notebook establishes, on `fusion_c_fused`, both splits, three seeds:

1. **Calibrator choice under family shift.** Isotonic wins in-distribution;
   Platt transfers better to unseen families. Platt is primary because
   family-disjoint is the deployment-honest condition.
2. **Regime-conditional calibration, selected on validation only.** A
   regime-specific map is adopted only where validation cross-validation shows
   a decisive (>= 2x) ECE gain. Result: per-regime for certificate-holders,
   global for no-certificate domains - the *hybrid* scheme. The test set then
   confirms it, and shows why validation could not have chosen otherwise.
3. **Operating points and the deferral band.** Thresholds derived on
   validation, reported on test: pass / defer to the cryptographic check /
   block. Plus the operator's single knob - block-zone FPR target versus
   deferral width.
4. **The saved trust score**: hybrid-calibrated `fusion_c_fused`, one artifact
   per run, read by notebooks 10 and 11.

In [ ]:
# --- standard header ---
from google.colab import drive
drive.mount('/content/drive')

import os, sys, subprocess, getpass
REPO = '/content/secure-dns-trust-ai'
URL  = 'github.com/sandesh20lamichhane/secure-dns-trust-ai.git'
if os.path.isdir(REPO):
    subprocess.run(['git','-C',REPO,'pull','-q'], check=False)
else:
    TOKEN = getpass.getpass('GitHub PAT: ')
    subprocess.run(['git','clone','-q',f'https://{TOKEN}@{URL}',REPO], check=True)
sys.path.insert(0, REPO)
os.environ['DNSTRUST_CONFIG_DIR'] = f'{REPO}/configs'

from src.utils import config, manifest, seeds
P = config.paths(); config.ensure_tree(P); seeds.set_all(42)
print('repo', manifest.git_sha(REPO))

In [ ]:
!pip -q install pyarrow zstandard scikit-learn

In [ ]:
import pandas as pd, numpy as np, json
from pathlib import Path
from sklearn.metrics import roc_curve
from sklearn.model_selection import StratifiedKFold
from src.evaluate import metrics, predictions
from src.models.calibrate import Calibrator, reliability_curve
from src.utils import manifest as mf

PRED_DIR = Path(P['artifacts']['predictions']); TAB = Path(P['results']['tables'])
TAB.mkdir(parents=True, exist_ok=True)
PRIMARY = 'fusion_c_fused'
SPLITS, SEEDS = ['family_disjoint_v1', 'random_v1'], [42, 43, 44]
BEST = 'platt'            # decided in section 1; Platt transfers under family shift
GAIN_RULE = 2.0           # per-regime map adopted only if validation-CV ECE improves >= 2x

_HC = pd.read_parquet(f"{P['data']['features']}/fused_v1.parquet",
                      columns=['domain', 'has_certificate'])

def load_pair(split_name, seed):
    te = predictions.load(f'{PRIMARY}_{split_name}_s{seed}', PRED_DIR)
    va = predictions.load(f'{PRIMARY}_{split_name}_s{seed}_VAL', PRED_DIR)
    va = va.merge(_HC, on='domain', how='left')
    te['has_certificate'] = te['has_certificate'].astype(bool)
    va['has_certificate'] = va['has_certificate'].astype(bool)
    return va, te

va, te = load_pair('family_disjoint_v1', 42)
print('val', va.shape, '| test', te.shape)
print('test regimes: cert', int(te.has_certificate.sum()), '| nocert', int((~te.has_certificate).sum()))

## 1. Calibrator choice

ROC is unchanged by any monotone calibration; Brier and ECE decide.

In [ ]:
rows = []
for split_name in SPLITS:
    for seed in SEEDS:
        va, te = load_pair(split_name, seed)
        for method in ['none', 'platt', 'isotonic']:
            cal = Calibrator(method).fit(va['raw_score'], va['true_label'])
            m = metrics.evaluate(te['true_label'], cal.transform(te['raw_score']))
            rows.append({'split': split_name, 'seed': seed, 'method': method,
                         'brier': m['brier_score'], 'ece': m['ece'], 'mce': m['mce'], 'roc_auc': m['roc_auc']})
cal_tab = (pd.DataFrame(rows).groupby(['split', 'method'])[['brier', 'ece', 'mce', 'roc_auc']]
             .agg(['mean', 'std']).round(4))
display(cal_tab); cal_tab.to_csv(TAB/'table_calibration_methods.csv')
print(f'PRIMARY calibrator: {BEST}  (Platt: better Brier/MCE under family shift; '
      'isotonic: best in-distribution - both reported)')

## 2. Regime-conditional calibration

### 2a. Scheme selection on validation only

For each regime, 5-fold CV inside the validation part compares a *global* map
(fitted on all validation rows outside the held-out fold) with a
*regime-specific* map (fitted on that regime's rows outside the fold). A
regime-specific map is adopted only where the CV-ECE gain is >= 2x. The test
set plays no part in this choice.

In [ ]:
rows = []
for split_name in SPLITS:
    for seed in SEEDS:
        va, _ = load_pair(split_name, seed)
        for flag, reg in [(False, 'nocert'), (True, 'cert')]:
            vr = va[va.has_certificate == flag].reset_index(drop=True)
            skf = StratifiedKFold(5, shuffle=True, random_state=seed)
            e_g, e_r = [], []
            for tr_i, te_i in skf.split(vr, vr['true_label']):
                held = set(vr.iloc[te_i]['domain'])
                va_tr = va[~va['domain'].isin(held)]
                g = Calibrator(BEST).fit(va_tr['raw_score'], va_tr['true_label'])
                r = Calibrator(BEST).fit(vr.iloc[tr_i]['raw_score'], vr.iloc[tr_i]['true_label'])
                y, x = vr.iloc[te_i]['true_label'].values, vr.iloc[te_i]['raw_score']
                e_g.append(metrics.expected_calibration_error(y, g.transform(x)))
                e_r.append(metrics.expected_calibration_error(y, r.transform(x)))
            rows.append({'split': split_name, 'seed': seed, 'regime': reg,
                         'cv_ece_global': np.mean(e_g), 'cv_ece_per_regime': np.mean(e_r)})
sel_raw = pd.DataFrame(rows)
sel = sel_raw.groupby(['split', 'regime'])[['cv_ece_global', 'cv_ece_per_regime']].agg(['mean', 'std']).round(4)
sel[('gain_factor', '')] = (sel[('cv_ece_global', 'mean')] / sel[('cv_ece_per_regime', 'mean')]).round(2)
sel[('selected', '')] = np.where(sel[('gain_factor', '')] >= GAIN_RULE, 'per_regime', 'global')
display(sel); sel.to_csv(TAB/'table_calibration_scheme_selection.csv')

SCHEME = {}
for (split_name, reg), r in sel.iterrows():
    SCHEME[(split_name, reg)] = r[('selected', '')]
print('SCHEME chosen on validation:', SCHEME)

### 2b. Confirmation on test

ECE and Brier inside each regime, for global, per-regime, and the *selected*
hybrid. This table is confirmation, not selection.

In [ ]:
def hybrid_calibrate(va, te, split_name):
    """Apply the validation-selected scheme per regime."""
    g = Calibrator(BEST).fit(va['raw_score'], va['true_label'])
    out = g.transform(te['raw_score'])
    for flag, reg in [(False, 'nocert'), (True, 'cert')]:
        if SCHEME[(split_name, reg)] == 'per_regime':
            vm, tm = (va.has_certificate == flag).values, (te.has_certificate == flag).values
            c = Calibrator(BEST).fit(va.loc[vm, 'raw_score'], va.loc[vm, 'true_label'])
            out[tm] = c.transform(te.loc[tm, 'raw_score'])
    return out

rows = []
for split_name in SPLITS:
    for seed in SEEDS:
        va, te = load_pair(split_name, seed)
        g = Calibrator(BEST).fit(va['raw_score'], va['true_label'])
        p_global = g.transform(te['raw_score'])
        p_regime = np.empty(len(te))
        for flag in (False, True):
            vm, tm = (va.has_certificate == flag).values, (te.has_certificate == flag).values
            c = Calibrator(BEST).fit(va.loc[vm, 'raw_score'], va.loc[vm, 'true_label'])
            p_regime[tm] = c.transform(te.loc[tm, 'raw_score'])
        p_hybrid = hybrid_calibrate(va, te, split_name)
        for scheme, p in [('global', p_global), ('per_regime', p_regime), ('hybrid_selected', p_hybrid)]:
            for reg, mask in [('all', np.ones(len(te), bool)),
                              ('nocert', ~te.has_certificate.values), ('cert', te.has_certificate.values)]:
                y = te.loc[mask, 'true_label'].values
                if len(np.unique(y)) < 2: continue
                m = metrics.evaluate(y, p[mask])
                rows.append({'split': split_name, 'seed': seed, 'scheme': scheme, 'regime': reg,
                             'ece': m['ece'], 'brier': m['brier_score'], 'mce': m['mce']})
regime_tab = (pd.DataFrame(rows).groupby(['split', 'regime', 'scheme'])[['ece', 'brier', 'mce']]
                .agg(['mean', 'std']).round(4))
display(regime_tab); regime_tab.to_csv(TAB/'table_calibration_per_regime.csv')

## 3. Reliability curves (data for figures)

In [ ]:
for split_name in SPLITS:
    va, te = load_pair(split_name, 42)
    p_raw, p_cal = te['raw_score'].values, hybrid_calibrate(va, te, split_name)
    for name, p in [('raw', p_raw), ('calibrated', p_cal)]:
        for reg, mask in [('all', np.ones(len(te), bool)),
                          ('nocert', ~te.has_certificate.values), ('cert', te.has_certificate.values)]:
            curve = pd.DataFrame(reliability_curve(te.loc[mask, 'true_label'].values, p[mask], n_bins=15))
            curve.to_csv(TAB/f'reliability_{split_name}_{name}_{reg}.csv', index=False)
print('reliability curves written')

## 4. Operating points and the deferral band

Thresholds are chosen on **validation** (calibrated) and reported on test:

* below `t_low` (validation TPR reaches 95%): **pass** - low suspicion;
* above `t_high` (validation FPR <= 0.1%): **block** - DANE/TLSA validation
  mandatory before any trust;
* between them: **defer** - the score abstains; the deterministic
  cryptographic check decides.

`defer_cert_share` is the diagnostic: if deferral concentrates on domains
without a certificate, the score abstains exactly where cryptography is cheap
and decisive (no certificate, no TLSA, no trust).

In [ ]:
def thresholds_from_val(scores, y, target_tpr=0.95, target_fpr=0.001):
    fpr, tpr, thr = roc_curve(y, scores)
    i_hr = np.argmax(tpr >= target_tpr)
    ok = np.where(fpr <= target_fpr)[0]
    i_lf = ok[np.argmax(tpr[ok])] if len(ok) else 0
    return float(thr[i_hr]), float(thr[i_lf])

def zones(va, te, split_name, target_fpr=0.001):
    pv = hybrid_calibrate(va, va, split_name)      # calibrate val with maps fitted on val
    pt = hybrid_calibrate(va, te, split_name)
    t_hr, t_lf = thresholds_from_val(pv, va['true_label'].values, target_fpr=target_fpr)
    lo, hi = min(t_hr, t_lf), max(t_hr, t_lf)
    y = te['true_label'].values
    pass_, defer, block = pt < lo, (pt >= lo) & (pt <= hi), pt > hi
    return dict(t_low=lo, t_high=hi,
        pass_frac=pass_.mean(), defer_frac=defer.mean(), block_frac=block.mean(),
        pass_malicious_rate=y[pass_].mean() if pass_.any() else np.nan,
        block_benign_rate=1 - y[block].mean() if block.any() else np.nan,
        defer_malicious_rate=y[defer].mean() if defer.any() else np.nan,
        recall_block_only=(block & (y == 1)).sum() / max((y == 1).sum(), 1),
        recall_block_plus_defer=((block | defer) & (y == 1)).sum() / max((y == 1).sum(), 1),
        defer_cert_share=te.loc[defer, 'has_certificate'].mean() if defer.any() else np.nan)

rows = []
for split_name in SPLITS:
    for seed in SEEDS:
        va, te = load_pair(split_name, seed)
        rows.append({'split': split_name, 'seed': seed, **zones(va, te, split_name)})
op = pd.DataFrame(rows)
op_tab = op.groupby('split').agg(['mean', 'std']).round(4)
display(op_tab[['t_low', 't_high', 'pass_frac', 'defer_frac', 'block_frac']])
display(op_tab[['pass_malicious_rate', 'block_benign_rate', 'defer_malicious_rate',
                'recall_block_only', 'recall_block_plus_defer', 'defer_cert_share']])
op.to_csv(TAB/'table_operating_points.csv', index=False)

### 4b. The operator's knob

Relaxing the block-zone FPR target narrows the deferral band at the cost of
benign collateral in the block zone. Recall through block-plus-defer is set by
`t_low` and does not move.

In [ ]:
rows = []
for split_name in SPLITS:
    for tf in (0.001, 0.005, 0.01, 0.02):
        for seed in SEEDS:
            va, te = load_pair(split_name, seed)
            z = zones(va, te, split_name, target_fpr=tf)
            rows.append({'split': split_name, 'target_fpr': tf, 'defer_frac': z['defer_frac'],
                         'block_benign_rate': z['block_benign_rate'],
                         'recall_block_plus_defer': z['recall_block_plus_defer']})
sweep = pd.DataFrame(rows).groupby(['split', 'target_fpr']).mean().round(4)
display(sweep); sweep.to_csv(TAB/'table_deferral_sweep.csv')

## 5. Save the trust score

Hybrid-calibrated `fusion_c_fused`, per run, with raw and calibrated columns.
This is the artifact notebooks 10 and 11 read.

In [ ]:
for split_name in SPLITS:
    for seed in SEEDS:
        va, te = load_pair(split_name, seed)
        cal = hybrid_calibrate(va, te, split_name)
        run_id = f'trustscore_{split_name}_s{seed}'
        predictions.save(run_id, PRED_DIR, te['domain'].values, te['true_label'].values,
                         te['raw_score'].values, calibrated_score=cal,
                         extra={'has_certificate': te['has_certificate'].values})
        m = metrics.evaluate(te['true_label'], cal)
        mf.record(P['manifest'], run_id, 'trustscore',
                  {'source_run': f'{PRIMARY}_{split_name}_s{seed}', 'calibrator': BEST,
                   'scheme': {k[1]: v for k, v in SCHEME.items() if k[0] == split_name},
                   'selection_rule': f'per-regime only if validation-CV ECE gain >= {GAIN_RULE}x'},
                  split_name, None, m, seed, repo_root=REPO)
        print(f'{run_id:34s} ECE={m["ece"]:.4f} Brier={m["brier_score"]:.4f} '
              f'scheme={ {k[1]: v for k, v in SCHEME.items() if k[0] == split_name} }')

---

**What the paper takes from here.** (1) Platt over isotonic under family shift,
with the in-distribution reversal reported. (2) The hybrid scheme, chosen by a
validation-only rule; the no-cert margin was too small to act on, and the
family-disjoint test shows why - validation contains no unseen families and
cannot anticipate that shift. (3) A deferral band that lands almost entirely
on no-certificate domains, where the cryptographic check is decisive by
default. (4) One operator knob with a measured trade-off.

**Next:** `10_xai_evaluation`.